<a href="https://colab.research.google.com/github/kuds/rl-drone/blob/main/%5BDrone%20Racer%5D%20Soft%20Actor-Critic%20(SAC).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Drone Track] Soft Actor-Critic (SAC)


In [ ]:
!pip install "rl-drone[train,curve] @ git+https://github.com/kuds/rl-drone.git"

# Set up GPU rendering.
import distutils.util
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

# Check if installation was succesful.
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Graphics and plotting.
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy

from IPython.display import clear_output
clear_output()

In [ ]:
# --- Standard Library ---
import os
import platform
import shutil
from datetime import datetime
from importlib.metadata import version

# --- Third-Party ---
import numpy as np
import torch

# --- rl_drone package ---
from rl_drone.envs.drone_racer import DroneRacerEnv
from rl_drone.utils.model_xml import setup_mujoco_model
from rl_drone.utils.track import (
    generate_equidistant_points,
    add_radial_noise_to_points_rng,
)
from rl_drone.utils.curve import Curve3D
from rl_drone.utils.plotting import plot_learning_curves
from rl_drone.callbacks import (
    ConfigSaveCallback,
    ReformatEvalCallback,
    TrainingPlotsCallback,
    VecNormalizeSaveCallback,
    VideoRecordCallback,
)

# --- Stable Baselines3 ---
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import (
    CallbackList,
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import (
    DummyVecEnv,
    VecNormalize,
    VecVideoRecorder,
)

# --- Environment Specific (Colab) ---
try:
    import google.colab.drive
except ImportError:
    pass  # Gracefully handle if not running in Colab

np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [ ]:
print(f"Python Version: {platform.python_version()}")
print(f"Gymnasium Version: {version('gymnasium')}")
print(f"Robot Description Version: {version('robot_descriptions')}")
print(f"Numpy Version: {version('numpy')}")
print(f"Mujoco Version: {version('mujoco')}")
print(f"Stable-Baselines3 Version: {version('stable-baselines3')}")
print(f"Matplotlib Version: {version('matplotlib')}")
print(f"Torch Version: {version('torch')}")
print(f"Is Cuda Available: {torch.cuda.is_available()}")
print(f"Cuda Version: {torch.version.cuda}")
if torch.cuda.is_available(): print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
rl_type = "SAC"
project_name = "BitCrazy"
env_str = "DroneRacer"
name_prefix = f"{env_str}_{rl_type}".lower()
log_dir = ""
use_google_drive = True
if use_google_drive:
    parent_path = "/content/gdrive"
    google.colab.drive.mount(parent_path, force_remount=True)
    log_dir = "{}/MyDrive/Finding Theta/{}/logs/{}/{}".format(parent_path,
                                                              project_name,
                                                              env_str,
                                                              rl_type)
else:
    log_dir = "/content/logs/{}/{}".format(env_str, rl_type)

training_data_path = os.path.join(log_dir, "training jobs")
time_folder = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
model_folder_path = os.path.join(log_dir, "training jobs", time_folder)

In [ ]:
env_config = {
    "frame_stack": 4,
    "episode_length": 5_000,
    "sphere_size": 0.1,
    "track_size": 1,
    "number_of_checkpoints": 12,
    "reward_function": "multiplicative_inverse",
    "terminate_without_contact": 450,
    "speed_factor": 0.25,
    "track_height": 0.25,
    "max_distance": 0.5,
    "time_penalty": 0.1,
    "out_of_bounds_penalty": -100,
    "no_contact_penalty": -100,
    "complete_bonus": 250,
    "contact_bonus": 25
}

model_config = {
    "eval_freq": 25_000,
    "n_envs": 4,
    "total_timesteps": 10_000_000,
    "n_eval_episodes": 25
}

In [ ]:
# Create Folders
os.makedirs(log_dir, exist_ok=True)
os.makedirs(training_data_path, exist_ok=True)
os.makedirs(model_folder_path, exist_ok=True)

In [ ]:
setup_mujoco_model(
    sphere_size=env_config["sphere_size"],
    target_height=env_config["track_height"],
)

In [ ]:
# --- Track generation and visualization ---
CIRCLE_RADIUS = env_config["track_size"]
NUM_POINTS = env_config["number_of_checkpoints"]
TRACK_HEIGHT = env_config["track_height"]
z_noise_scale = 0.25
angle_noise_scale = 0.25

equidistant_points = generate_equidistant_points(CIRCLE_RADIUS, NUM_POINTS, TRACK_HEIGHT)

print(f"Generated {NUM_POINTS} equidistant points on a circle with radius {CIRCLE_RADIUS}:")
for i, p in enumerate(equidistant_points):
    print(f"  Point {i:02d}: (X={p[0]:.2f}, Y={p[1]:.2f})")

noisy_coords = add_radial_noise_to_points_rng(
    equidistant_points, angle_noise_scale, z_noise_scale, skip_origin=True, seed=0
)
print(f"Generated {NUM_POINTS} equidistant noisy points on a circle with radius {CIRCLE_RADIUS}:")
for i, p in enumerate(noisy_coords):
    print(f"  Point {i:02d}: (X={p[0]:.2f}, Y={p[1]:.2f}, Z={p[2]:.2f})")

noisy_curve = Curve3D(noisy_coords, closed=True, num_samples=200)

print(f"\nCreated curve through {len(noisy_coords)} points")
print(f"Curve length: {noisy_curve.get_length():.4f}")
print(f"Curve is closed: {noisy_curve.closed}")

noisy_curve.visualize()

In [ ]:
env = DroneRacerEnv(env_config)
print("Observation Space Size: ", env.observation_space.shape)
print('Actions Space: ', env.action_space)
env.close()

In [ ]:
def make_env():
  env = DroneRacerEnv(render_mode="rgb_array", env_config=env_config)
  check_env(env)
  return env

In [ ]:
# Create Training environment
env = make_vec_env(make_env,
                   n_envs=model_config["n_envs"],
                   monitor_dir=os.path.join(model_folder_path, "monitor"))

normalized_env = VecNormalize(env,
                              training=True,
                              norm_obs=True,
                              norm_reward=True)


# Create Evaluation environment
env_val = make_vec_env(make_env, n_envs=1)
normalized_env_val = VecNormalize(env_val,
                                  training=False,
                                  norm_obs=True,
                                  norm_reward=True)

checkpoint_callback = CheckpointCallback(
  save_freq=model_config["eval_freq"],
  save_path=os.path.join(model_folder_path, "checkpoints"),
  name_prefix="rl_model",
  save_replay_buffer=False,
  save_vecnormalize=True,
)

reformat_callback = ReformatEvalCallback(
    save_path=model_folder_path,
    eval_file=os.path.join(model_folder_path, "evaluations.npz"),
    save_freq=model_config["eval_freq"],
    csv_file_name="SummaryEvaluations",
)

vec_normalize_callback = VecNormalizeSaveCallback(save_path=model_folder_path)

config_save_callback = ConfigSaveCallback(
    save_path=model_folder_path,
    hyperparams={"env_config": env_config, "model_config": model_config},
    run_name=f"{env_str}_{rl_type}",
)

training_plots_callback = TrainingPlotsCallback(
    eval_file=os.path.join(model_folder_path, "evaluations.npz"),
    save_path=model_folder_path,
    save_freq=model_config["eval_freq"],
)

video_record_callback = VideoRecordCallback(
    make_env_fn=make_env,
    save_path=os.path.join(model_folder_path, "videos"),
    video_length=10_000,
    save_freq=model_config["eval_freq"],
    name_prefix=name_prefix,
)

eval_callback = EvalCallback(normalized_env_val,
                             best_model_save_path=model_folder_path,
                             log_path=model_folder_path,
                             render=False,
                             deterministic=True,
                             callback_on_new_best=vec_normalize_callback,
                             n_eval_episodes=model_config["n_eval_episodes"],
                             eval_freq=model_config["eval_freq"])

# Create the callback list
callbackList = CallbackList([
    checkpoint_callback,
    eval_callback,
    reformat_callback,
    config_save_callback,
    training_plots_callback,
    video_record_callback,
])

# learning with tensorboard logging and saving model
model = SAC("MlpPolicy",
            normalized_env,
            verbose=0,
            tensorboard_log=os.path.join(log_dir, "tensorboard"))

model.learn(total_timesteps=model_config["total_timesteps"],
            callback=callbackList,
            progress_bar=False)

# Save the model
model.save(os.path.join(model_folder_path, "final_model"))

mean_reward, std_reward = evaluate_policy(model,
                                          normalized_env_val,
                                          deterministic=True,
                                          n_eval_episodes=model_config["n_eval_episodes"])

print(f"Final Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

normalized_env.save(os.path.join(model_folder_path, "final_normalized_env.pkl"))
normalized_env_val.save(os.path.join(model_folder_path, "final_normalized_env_val.pkl"))

env.close()
env_val.close()
normalized_env.close()
normalized_env_val.close()

In [ ]:
# Create Evaluation environment
env_val = make_vec_env(make_env, n_envs=1, seed=0)
normalized_env_val = VecNormalize.load(os.path.join(model_folder_path, "best_model_vec_normalize.pkl"),
                                       venv=env_val)

# Load the best model
best_model_path = os.path.join(model_folder_path, "best_model")
best_model = SAC.load(best_model_path, env=normalized_env_val)

mean_reward, std_reward = evaluate_policy(best_model,
                                          normalized_env_val,
                                          deterministic=True,
                                          n_eval_episodes=model_config["n_eval_episodes"])

print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

# Close the environment
env_val.close()
normalized_env_val.close()

In [ ]:
# Record video of the best model
video_length = 10_000
video_folder_path = os.path.join(model_folder_path, "videos")
os.makedirs(video_folder_path, exist_ok=True)
for i in range(5):
    env_val = make_vec_env(make_env, n_envs=1)
    normalized_env_val = VecNormalize.load(
        os.path.join(model_folder_path, "best_model_vec_normalize.pkl"),
        venv=env_val,
    )

    best_model_path = os.path.join(model_folder_path, "best_model")
    best_model = SAC.load(best_model_path, env=normalized_env_val)
    best_model_file_name = f"best_model_{name_prefix}_{i}"
    env = VecVideoRecorder(
        normalized_env_val,
        video_folder_path,
        video_length=video_length,
        record_video_trigger=lambda x: x == 0,
        name_prefix=best_model_file_name,
    )

    obs = env.reset()
    total_reward = 0.0
    session_length = 0
    for _ in range(video_length):
        action, _states = best_model.predict(obs, deterministic=True)
        obs, rewards, done, info = env.step(action)
        total_reward += rewards
        env.render()
        session_length += 1
        if done:
            break

    print(f"Playback {i+1}: steps={session_length}, reward={total_reward[0]:.2f}")

    env.close()
    env_val.close()
    normalized_env_val.close()

In [ ]:
plot_learning_curves(
    eval_file=os.path.join(model_folder_path, "evaluations.npz"),
    save_dir=model_folder_path,
    name_prefix=f"{rl_type}_{env_str}",
    show=True,
)

In [ ]:
# Save a copy of this notebook to the model folder
notebook_name = "notebook.ipynb"
%notebook -e $notebook_name

source_file = os.path.join(notebook_name)
destination_file = os.path.join(model_folder_path, notebook_name)

try:
    shutil.copyfile(notebook_name, destination_file)
    print(f"File '{source_file}' copied to '{destination_file}' successfully.")
except FileNotFoundError:
    print(f"Error: Source file '{source_file}' not found.")
except shutil.SameFileError:
    print(f"Error: Source and destination are the same file.")
except Exception as e:
    print(f"An error occurred: {e}")